# 4f Optical Correlator

Notebook version of `scripts/4f_correlator.py`.  We walk through
the physics of a 4f optical correlator that uses a **matched filter** to
locate copies of a target pattern inside a larger scene.

### Outline

0. **Imports** for optics components and numerics.
1. **Paths and Parameters** for saved artifacts.
2. **Helper functions** with physics background (Fourier-transforming lens,
   sampling-matched focal length, matched filter, paraxial validity).
3. **Setup** of the 4f correlator optical system.
4. **Evaluation** and comparison with direct FFT cross-correlation.
5. **Plot Results** for visual inspection.


## 0  Imports

In [ ]:
from __future__ import annotations

from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from fouriax.optics import (
    ComplexMask,
    DetectorArray,
    Field,
    Grid,
    OpticalModule,
    Spectrum,
    ThinLens,
    plan_propagation,
)

# NOTEBOOK_REPO_ROOT_SETUP
import os
from pathlib import Path as _Path
%matplotlib inline

def _find_repo_root(start: _Path) -> _Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "src" / "fouriax").exists()
            and (candidate / "README.md").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root from current working directory. "
        "Expected to find src/fouriax and README.md in an ancestor."
    )

REPO_ROOT = _find_repo_root(_Path.cwd())
if _Path.cwd() != REPO_ROOT:
    os.chdir(REPO_ROOT)


## 1  Paths and Parameters

In [ ]:
ARTIFACTS_DIR = Path("artifacts")
PLOT_PATH = ARTIFACTS_DIR / "4f_correlator.png"

WAVELENGTH_UM = 0.532
N_MEDIUM = 1.0
GRID_N = 128
GRID_DX_UM = 2.0


## 2  Helper Functions

### Fourier-transforming property of a lens

A coherent field $E_{\text{in}}(x)$ placed at the **front focal plane** of
a thin lens of focal length $f$ produces, at the **back focal plane**, the
field

$$
E_{\text{FP}}(x') = \frac{1}{i\lambda f}\,
\int E_{\text{in}}(x)\,
\exp\!\Bigl(-i\,\frac{2\pi}{\lambda f}\,x\,x'\Bigr)\,dx
= \frac{1}{i\lambda f}\;\tilde{E}_{\text{in}}\!\!\left(\frac{x'}{\lambda f}\right).
$$

In words: the output is the **spatial Fourier transform** of the input,
evaluated at spatial frequency $f_x = x'/(\lambda f)$, up to a constant
prefactor.

Crucially, the input must be at the front focal plane — a distance $f$
before the lens — for this to hold *without* a residual quadratic-phase
term.  The complete path for one Fourier-transforming stage is therefore

$$
\boxed{\text{prop}(f) \;\to\; \text{Lens}(f) \;\to\; \text{prop}(f)}
$$

### Sampling-matched focal length

We simulate on a discrete $N \times N$ grid with pixel spacing $\Delta x$.
The highest spatial frequency the grid can represent is
$f_{x,\max} = 1/(2\Delta x)$ (the Nyquist frequency).  At the Fourier
plane, this frequency maps to position

$$
x'_{\max} = \lambda f \, f_{x,\max} = \frac{\lambda f}{2\,\Delta x}.
$$

For the Fourier-plane grid to have the **same pixel spacing** $\Delta x$
and the **same number of samples** $N$, we need

$$
x'_{\max} = \frac{N\,\Delta x}{2}
\qquad\Longrightarrow\qquad
\boxed{f = \frac{n\,N\,\Delta x^2}{\lambda}},
$$

where $n$ is the refractive index of the medium.  This is the
**sampling-matched focal length**.  It guarantees a one-to-one mapping
between the DFT frequency bins and the physical positions at the Fourier
plane.

### The matched filter

A 4f system places two Fourier-transforming stages back to back with a
filter mask at the shared Fourier plane:

$$
\text{input}
\;\xrightarrow{\text{prop}(f)}\;
\text{Lens}_1
\;\xrightarrow{\text{prop}(f)}\;
\underbrace{H(f_x, f_y)}_{\text{filter}}
\;\xrightarrow{\text{prop}(f)}\;
\text{Lens}_2
\;\xrightarrow{\text{prop}(f)}\;
\text{output}
$$

Without the filter ($H = 1$), the two Fourier transforms compose to give
a spatially inverted copy of the input — a trivial imaging system.

With a matched filter

$$
H(f_x, f_y) = \tilde{T}^*(f_x, f_y)
= \mathcal{F}^*\{\text{target}\},
$$

the output field becomes (up to constants)

$$
E_{\text{out}} \propto \mathcal{F}^{-1}\!\bigl\{
\tilde{S}(f_x, f_y)\;\tilde{T}^*(f_x, f_y)\bigr\},
$$

which is exactly the **cross-correlation** of the scene $S$ with the
target $T$ (by the correlation theorem).  The output **intensity**
$|E_{\text{out}}|^2$ will show bright peaks wherever the target pattern
appears in the scene.

### Paraxial validity constraint

The thin-lens Fourier-transform relationship relies on the **paraxial
approximation**: rays make small angles with the optical axis.  The worst
case is a ray from the edge of the grid ($r_{\max} = N\Delta x / 2$) to
the lens centre, subtending an angle $\theta \approx r_{\max}/f$.

Using the sampling-matched focal length,

$$
\left(\frac{r_{\max}}{f}\right)^{\!2}
= \left(\frac{\lambda}{2\,n\,\Delta x}\right)^{\!2},
$$

which depends only on $\lambda$ and $\Delta x$ — *not* on $N$.  We need
this ratio $\ll 1$ for the paraxial approximation to hold.  Choosing
$\Delta x \gg \lambda/(2n)$ ensures this.


In [ ]:
def _raw_correlate(scene: jnp.ndarray, target: jnp.ndarray) -> jnp.ndarray:
    """Ground-truth cross-correlation intensity via direct FFT."""
    f_scene = jnp.fft.fftn(jnp.fft.ifftshift(scene), axes=(-2, -1))
    f_target = jnp.fft.fftn(jnp.fft.ifftshift(target), axes=(-2, -1))
    corr = jnp.fft.ifftn(f_scene * jnp.conj(f_target), axes=(-2, -1))
    return jnp.abs(jnp.fft.fftshift(corr)) ** 2


def _sampling_matched_focal_length(grid: Grid) -> float:
    """f = n·N·dx² / λ"""
    return N_MEDIUM * grid.nx * grid.dx_um**2 / WAVELENGTH_UM


def _matched_filter(target: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Fourier-plane matched filter: (amplitude, phase_rad).

    Built in the centred (fftshift) convention to match the physical
    Fourier plane, where DC sits at the optical axis.
    """
    ft = jnp.fft.fftshift(
        jnp.fft.fftn(jnp.fft.ifftshift(target), axes=(-2, -1)),
        axes=(-2, -1),
    )
    amplitude = jnp.abs(ft) / jnp.max(jnp.abs(ft))
    phase_rad = -jnp.angle(ft)
    return amplitude, phase_rad


def _paraxial_validity_constraint_fom() -> float:
    """Dimensionless paraxial figure-of-merit (r_max / f)^2 for this grid pitch."""
    return (WAVELENGTH_UM / (2 * N_MEDIUM * GRID_DX_UM)) ** 2


def _make_rect(grid: Grid, cx_f: float, cy_f: float, w_f: float, h_f: float) -> jnp.ndarray:
    """Binary rectangle; positions/sizes are fractions of grid half-extent."""
    half = grid.nx * grid.dx_um / 2.0
    x, y = grid.spatial_grid()
    return ((jnp.abs(x - cx_f * half) <= w_f * half) &
            (jnp.abs(y - cy_f * half) <= h_f * half)).astype(jnp.float32)


def _build_scene(grid: Grid) -> jnp.ndarray:
    """Three copies of the target square plus a distractor rectangle."""
    hw = 0.10
    scene = jnp.zeros(grid.shape, dtype=jnp.float32)
    for cx, cy in [(-0.35, -0.25), (0.20, 0.38), (0.40, -0.20)]:
        scene = scene + _make_rect(grid, cx, cy, hw, hw)
    scene = scene + 0.5 * _make_rect(grid, -0.18, 0.20, 0.18, 0.05)
    return jnp.clip(scene, 0.0, 1.0)


def _build_target(grid: Grid) -> jnp.ndarray:
    """Centred target square (same size as copies in the scene)."""
    return _make_rect(grid, 0.0, 0.0, 0.10, 0.10)


## 3  Setup

The optical system uses `ASMPropagator` (Angular Spectrum Method) with the
sampling planner disabled so that the grid spacing stays exactly matched
to the focal length.

The output is spatially inverted because two successive Fourier transforms
flip the image.  We flip it back for comparison with the ground truth.

In [ ]:
grid = Grid.from_extent(nx=GRID_N, ny=GRID_N, dx_um=GRID_DX_UM, dy_um=GRID_DX_UM)
spectrum = Spectrum.from_scalar(WAVELENGTH_UM)
f_um = _sampling_matched_focal_length(grid)
print(f"f = {f_um:.1f} µm  |  (r_max/f)² = "
      f"{_paraxial_validity_constraint_fom():.4f}")

scene = _build_scene(grid)
target = _build_target(grid)
field_in = Field.plane_wave(grid=grid, spectrum=spectrum).apply_amplitude(scene[None, :, :])

amp, phase = _matched_filter(target)

prop = plan_propagation(
    mode="auto",
    grid=grid,
    spectrum=spectrum,
    distance_um=f_um,
)
lens = ThinLens(focal_length_um=f_um)

correlator = OpticalModule(
    layers=(
        prop, lens, prop,
        ComplexMask(amplitude_map=amp, phase_map_rad=phase),
        prop, lens, prop,
    ),
    sensor=DetectorArray(detector_grid=grid),
)


## 4  Evaluation

In [ ]:
output_4f = np.asarray(correlator.measure(field_in))
output_raw = np.asarray(_raw_correlate(scene, target))

output_4f = output_4f[::-1, ::-1]
norm = lambda x: x / np.max(x) if np.max(x) > 0 else x  # noqa: E731
out_4f_n, out_raw_n = norm(output_4f), norm(output_raw)
cc = float(np.corrcoef(out_4f_n.ravel(), out_raw_n.ravel())[0, 1])
print(f"Correlation with ground truth: {cc:.4f}")


## 5  Plot Results

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(np.asarray(scene), cmap="gray")
axes[0].set_title("Input scene")
axes[1].imshow(np.asarray(target), cmap="gray")
axes[1].set_title("Target pattern")

for ax, img, title in zip(
    axes[2:],
    [out_4f_n, out_raw_n],
    [f"Physical 4f correlator (ρ = {cc:.4f})", "Raw FFT correlation"],
    strict=True,
):
    im = ax.imshow(img, cmap="hot")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(PLOT_PATH, dpi=150)
plt.show()
print(f"saved: {PLOT_PATH}")
